Documents

LangChain implements Document abstraction, which is intended to respresent a unit of text and asscociated metadata. It has two attributes

1. page_content: a string
2. metadata: a dict containing metadata. It has information about the source of the documents, its relationship to other doucment, etc.

In [5]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = model = ChatGroq(model="gemma2-9b-it", groq_api_key=groq_api_key)
llm


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000020558E9C0A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020558E69AB0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

c:\Notes\ML-DL-GenAI\LangChain\virtual_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nadeem Chaudhary\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not instal

In [7]:
# Vectore Store
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents, embedding=embeddings)
vectorstore

c:\Notes\ML-DL-GenAI\LangChain\virtual_env\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [8]:
vectorstore.similarity_search("Cat")

c:\Notes\ML-DL-GenAI\LangChain\virtual_env\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[Document(id='144c54e1-8427-4504-8682-1e0fbde7eed7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='e77ee102-3c78-4b00-adb7-1bda636f07c1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='4da1a04e-bd2d-457a-bf6b-dfc4be1d01c7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='dccbe4c2-a472-4a6a-8618-b2e958877c9a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [9]:
#Async Query
await vectorstore.asimilarity_search("cat")

c:\Notes\ML-DL-GenAI\LangChain\virtual_env\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[Document(id='144c54e1-8427-4504-8682-1e0fbde7eed7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='e77ee102-3c78-4b00-adb7-1bda636f07c1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='4da1a04e-bd2d-457a-bf6b-dfc4be1d01c7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='dccbe4c2-a472-4a6a-8618-b2e958877c9a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

Retrivers

LangChain VectoreStore objects do not subclass Runnable and so cannot immediately be integrated into LangChain Expression Language Chains.

LangChain Retrivers are Runnable so they implement a standard set of methods (eg. synchronous and asynchronous invoke and batch operations) and are desgiend to be incorporated in LCEL chains

In [10]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriver=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriver.batch(["Cat", "Dog"])

c:\Notes\ML-DL-GenAI\LangChain\virtual_env\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[[Document(id='144c54e1-8427-4504-8682-1e0fbde7eed7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='4da1a04e-bd2d-457a-bf6b-dfc4be1d01c7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the provided context only
{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([('human', message)])
rag_chain = {"context":retriver, "question":RunnablePassthrough()}|prompt|llm
response=rag_chain.invoke("Tell me about dog")
print(response.content)

c:\Notes\ML-DL-GenAI\LangChain\virtual_env\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Dogs are great companions, known for their loyalty and friendliness. 

